# GTM Chemical Space Analysis: Mapping ChEMBL Ligands and Your Target Compounds

This notebook uses **Generative Topographic Mapping (GTM)** to visualise the chemical space of
bioactive compounds from ChEMBL alongside your own screening or experimental dataset. The result
is a 2-D activity landscape that shows:
- Where your compounds sit in the known chemical space of a given target.
- Regions of high / low potency on the ChEMBL reference map.
- How similar (or novel) your compounds are relative to published actives.

---

## What is GTM?

GTM is a probabilistic, non-linear dimensionality reduction method originally developed by Bishop
et al. (1998). It maps a high-dimensional fingerprint space to a regular 2-D grid of latent nodes,
each associated with a Gaussian kernel in fingerprint space. Training finds the kernel positions
and a global precision parameter β that best explain the data via maximum likelihood (EM algorithm).

**Why use it for drug discovery?**

| Feature | GTM advantage |
|---------|--------------|
| Probabilistic projection | Every molecule gets a *responsibility* vector — a soft assignment to grid nodes rather than a single point. |
| Fixed reference frame | Train once on a large reference library (e.g. ChEMBL actives); project new compounds **without retraining**. This avoids training bias. |
| Activity landscapes | Colour the 2-D map by pIC50 / pKi to see SAR trends at a glance. |
| Uncertainty quantification | Responsibility entropy flags molecules that are poorly represented on the map. |

**Key workflow concept:**  
We train GTM on ChEMBL data for a reference chemical space, then project *your* compounds onto the
same map. Because we do not retrain, the map is not biased towards your dataset — your compounds
either fall inside the known chemical space (low entropy, reliable projection) or outside it
(high entropy, novel scaffold).

---

## Prerequisites

```bash
pip install gtm-chem rdkit-pypi pandas numpy matplotlib scipy
```

Or, if `gtm-chem` is installed from source:
```bash
git clone https://github.com/<your-org>/gtm_chem.git
cd gtm_chem && pip install -e .
```

---

## Required inputs

| Variable | Description |
|----------|-------------|
| `CHEMBL_CSV` | Path to ChEMBL activity CSV exported from ChEMBL for your target |
| `MY_DATA_CSV` | Path to your own compound CSV (SMILES + activity column) |
| `ACTIVITY_CUTOFF` | IC50 threshold (nM) for active/inactive split in ChEMBL deduplication |
| `MY_SMILES_COL` | Column name for SMILES in your CSV |
| `MY_ACTIVITY_COL`| Column name for activity (pIC50 / pKi) in your CSV |
| `MY_NAME_COL` | Column name for compound ID in your CSV |
| `OUTPUT_DIR` | Directory where figures and the trained model will be saved |

---

## How to obtain ChEMBL data

1. Go to [ChEMBL](https://www.ebi.ac.uk/chembl/).
2. Search for your target (e.g. *CYP3A4*, *EGFR*, *DRD2*).
3. Click **Activity** → filter by `standard_type = IC50` (or Ki / pIC50).
4. Export as CSV. The expected columns are shown in **Section 1** below.

## Section 0 — Configuration

Edit only this cell before running the entire notebook.

In [ ]:
# ============================================================
#  CONFIGURATION — edit these paths and parameters
# ============================================================

# ── Paths ────────────────────────────────────────────────────
CHEMBL_CSV     = "data/chembl_target_activity.csv"   # ChEMBL export CSV
MY_DATA_CSV    = "data/my_compounds.csv"             # your own data
OUTPUT_DIR     = "output/"                           # figures and model saved here

# ── Your dataset column names ───────────────────────────────
MY_NAME_COL     = "compound_id"    # unique compound identifier
MY_SMILES_COL   = "smiles"         # SMILES string column
MY_ACTIVITY_COL = "pIC50"          # activity column (pIC50, pKi, etc.)
MY_DATASET_LABEL = "My Compounds"  # label used in figure legends

# ── GTM hyperparameters (sensible defaults) ──────────────────
GTM_GRID_SIZE  = 30   # K = grid_size²; larger → finer map but slower training
GTM_RBF_SIZE   = 6    # number of RBF centres per axis; typically grid_size/5
GTM_N_ITER     = 300  # EM iterations
PCA_N_COMPONENTS = 50 # dimensionality fed into GTM

# ── ChEMBL deduplication ────────────────────────────────────
# Compounds with conflicting active/inactive labels are removed.
# Set this to the IC50 boundary (nM) that defines "active".
ACTIVITY_CUTOFF_NM = 1e5   # 100 µM; compounds above this are treated as inactive

# ── Fingerprint ─────────────────────────────────────────────
FP_TYPE = "ecfp4"   # options: ecfp4, ecfp6, fcfp4, rdkit, maccs, …
                    # run list_fingerprints() after imports to see all

# ── Optional: load a pre-trained model instead of retraining ─
# Set to None to train from scratch. Set to a .pkl path to skip training.
PRETRAINED_MODEL_PATH = None   # e.g. "output/chembl_ecfp4_gtm.pkl"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  ChEMBL CSV : {CHEMBL_CSV}")
print(f"  My data    : {MY_DATA_CSV}")
print(f"  Output dir : {OUTPUT_DIR}")

## Section 1 — Imports & Library Check

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ── gtm_chem ─────────────────────────────────────────────────────────────────
# If installed via pip, the import below should work directly.
# If running from a local clone, uncomment and adjust the path:
# sys.path.insert(0, "/path/to/gtm_chem")

from gtm_chem import (
    GTM,
    compute_fingerprints,
    variance_filter,
    pca_reduce,
    apply_pca,
    list_fingerprints,
    plot_landscape,
    plot_activity,
    plot_overlay,
    plot_multi_class,
    plot_uncertainty,
    plot_training_curve,
)

# ── RDKit ─────────────────────────────────────────────────────────────────────
from rdkit import Chem
from rdkit.Chem import rdmolops
from rdkit.Chem.SaltRemover import SaltRemover

print("All imports OK.")
print("\nAvailable fingerprint types:")
list_fingerprints()

## Section 2 — Load ChEMBL Data

The CSV exported from ChEMBL typically contains one row per activity measurement. A single compound
can appear multiple times (different assays, different labs). We will:
1. Inspect the raw data.
2. Deduplicate using a simple activity consensus rule.
3. Remove compounds with contradictory active/inactive labels — they are unreliable.

In [ ]:
# ── Load ChEMBL CSV ─────────────────────────────────────────────────────────
chembl_df = pd.read_csv(CHEMBL_CSV)

# Normalise column names: strip whitespace, replace spaces with underscores
chembl_df.columns = (
    chembl_df.columns
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
)

print(f"Raw ChEMBL rows loaded : {len(chembl_df):,}")
print()
chembl_df.info()
chembl_df.head(3)

### 2a — Deduplication and activity consensus

**Rule:**
- If all measurements for a compound are < `ACTIVITY_CUTOFF_NM` → keep one row (active).
- If all measurements are ≥ `ACTIVITY_CUTOFF_NM` → keep one row (inactive).
- If measurements are split across the boundary → **remove entirely** (conflicting data).

This is a conservative approach. Compounds with ambiguous activity profiles are better excluded
than treated as actives or inactives.

In [ ]:
def _filter_group(grp):
    """Consensus rule for one compound_chembl_id group."""
    if len(grp) == 1:
        return grp
    vals = grp['standard_value'].dropna()
    if len(vals) == 0:
        return grp.iloc[[0]]
    all_active   = (vals < ACTIVITY_CUTOFF_NM).all()
    all_inactive = (vals >= ACTIVITY_CUTOFF_NM).all()
    if all_active or all_inactive:
        return grp.iloc[[0]]
    # Mixed: remove compound
    cid = grp['compound_chembl_id'].iloc[0]
    print(f"  Removed (conflicting labels): {cid}")
    return None

filtered = (
    chembl_df
    .groupby('compound_chembl_id', group_keys=False)
    .apply(_filter_group)
)
chembl_df = filtered.reset_index(drop=True)
print(f"\nAfter deduplication: {len(chembl_df):,} rows")

### 2b — Keep only compounds with reliable activity values

We keep:
- Rows with `standard_relation = '='` (exact measurement, not >, <, ≥, ≤).
- Rows explicitly labelled `Not Active` / `inactive` in `activity_comment`.

Compounds with inequality relations (e.g. IC50 > 10 µM) are kept if they fall in the inactive
bucket, but their pChEMBL values are NaN and will appear as grey points on the map.

In [ ]:
chembl_filt = chembl_df[
    (chembl_df['standard_relation'] == '=') |
    (chembl_df.get('activity_comment', pd.Series(dtype=str))
               .isin(['Not Active', 'inactive', 'not active']))
].copy()
print(f"Rows with reliable activity: {len(chembl_filt):,}")

## Section 3 — SMILES Cleaning

Before computing fingerprints we:
1. **Remove salts / counterions** – keep only the largest organic fragment.
2. **Sanitise** – RDKit re-aromatises and validates the molecule.
3. **Size filter** – molecules > 50 heavy atoms are excluded (configurable); these are often
   macrocycles or polymer-like structures that create artefacts in ECFP fingerprints.

Adjust `MAX_HAC` if your target has large macrocyclic ligands (e.g. natural product targets).

In [ ]:
MAX_HAC = 60   # maximum heavy atom count; increase for macrocycle targets

_DEFAULT_SALT_SMARTS = [
    "[Li,Na,K,Rb,Ca,F,Cl,Br,I,O]", "O=C(O)C(F)(F)F",
    "CS(=O)(=O)O", "O=S(=O)(O)O", "O=C(O)C(O)C(O)C(=O)O",
    "CC(=O)[O-]", "O=C(O)C(=O)O", "O=S(=O)(O)CCO", "O=CO", "NCCO",
]
_SALT_REMOVERS = [SaltRemover(defnData=x) for x in _DEFAULT_SALT_SMARTS]


def clean_smiles(smi: str) -> str | None:
    """
    Parse, desalt, and canonicalise a SMILES string.
    Returns canonical SMILES or None if the molecule is invalid or too large.
    """
    if not isinstance(smi, str) or not smi.strip():
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    # Remove salts iteratively
    for remover in _SALT_REMOVERS:
        mol = remover.StripMol(mol)
    # Pick largest fragment (handles multi-component SMILES)
    frags = rdmolops.GetMolFrags(mol, asMols=True)
    if frags:
        mol = max(frags, key=lambda m: m.GetNumAtoms())
    if mol.GetNumAtoms() > MAX_HAC:
        return None
    return Chem.MolToSmiles(mol)


def build_mol_list(df, name_col, smi_col, activity_col):
    """
    Clean SMILES and return (names, smiles, activities) as parallel lists,
    skipping invalid / oversized molecules.
    """
    names, smiles_out, activities = [], [], []
    n_skip = 0
    for _, row in df.iterrows():
        csmi = clean_smiles(row[smi_col])
        if csmi is None:
            n_skip += 1
            continue
        names.append(row[name_col])
        smiles_out.append(csmi)
        activities.append(row.get(activity_col, np.nan))
    print(f"  Kept {len(smiles_out):,} / {len(df):,} molecules  ({n_skip} skipped)")
    return names, smiles_out, np.array(activities, dtype=float)

print("SMILES cleaning utilities defined.")

### 3a — Clean ChEMBL SMILES

In [ ]:
print("Cleaning ChEMBL SMILES …")
chembl_names, chembl_smi, chembl_activity = build_mol_list(
    chembl_filt,
    name_col     = 'compound_chembl_id',
    smi_col      = 'canonical_smiles',
    activity_col = 'pchembl_value',   # pIC50 column from ChEMBL
)
print(f"ChEMBL: {len(chembl_smi):,} clean molecules")

## Section 4 — Load Your Own Data

Your CSV must contain at minimum:
- A **SMILES column** (`MY_SMILES_COL`)
- An **activity column** (`MY_ACTIVITY_COL`) — pIC50, pKi, or any numeric activity
- A **name/ID column** (`MY_NAME_COL`)

Missing activity values are allowed; those compounds will appear as grey dots on the map.

In [ ]:
my_df = pd.read_csv(MY_DATA_CSV)
my_df.columns = (
    my_df.columns.astype(str).str.strip().str.replace(r'\s+', '_', regex=True)
)
print(f"Your data: {len(my_df):,} rows loaded")
print(f"Columns  : {list(my_df.columns)}")
my_df.head(3)

In [ ]:
# Optional: deduplicate by compound ID, averaging activity across replicates
my_df = (
    my_df
    .groupby(MY_NAME_COL, as_index=False)
    .agg({
        MY_ACTIVITY_COL: 'mean',
        **{c: 'first' for c in my_df.columns
           if c not in [MY_NAME_COL, MY_ACTIVITY_COL]}
    })
)
print(f"After deduplication: {len(my_df):,} unique compounds")

print("\nCleaning your SMILES …")
my_names, my_smi, my_activity = build_mol_list(
    my_df,
    name_col     = MY_NAME_COL,
    smi_col      = MY_SMILES_COL,
    activity_col = MY_ACTIVITY_COL,
)
print(f"Your data: {len(my_smi):,} clean molecules")

## Section 5 — Fingerprints, PCA and GTM Training

### Pipeline overview

```
SMILES
  └─ compute_fingerprints()  → binary bit-vector (e.g. ECFP4, 2048 bits)
       └─ variance_filter()  → remove near-constant bits (speeds up PCA)
            └─ pca_reduce()  → compress to PCA_N_COMPONENTS (default 50)
                 └─ GTM.fit_transform()  → 2D map coordinates + model
```

**Why PCA before GTM?**  
ECFP4 vectors are 2048-bit binary; many bits are near-zero across the entire library. PCA removes
redundant dimensions and produces a dense continuous representation that the EM algorithm in GTM
handles more efficiently. 50 principal components typically capture > 60–70% of variance for
drug-like compound libraries.

**GTM hyperparameters:**
- `grid_size` — number of latent nodes along each axis (total K = grid_size²). Default 30 → 900 nodes. Increase for very large, diverse libraries.
- `rbf_size` — RBF kernel width (number of RBF centres per axis). Controls the smoothness of the manifold.
- `n_iter` — EM iterations. 200–300 is usually sufficient for convergence.

If you have a pre-trained model (PRETRAINED_MODEL_PATH is set), we skip training and load it directly.
This is the recommended approach when projecting multiple target datasets onto the same reference map.

In [ ]:
# ── Step 1: Fingerprints ─────────────────────────────────────────────────────
print(f"Computing {FP_TYPE} fingerprints for {len(chembl_smi):,} ChEMBL molecules …")
X_fps, valid_mask = compute_fingerprints(chembl_smi, fp_type=FP_TYPE, return_valid_mask=True)
print(f"  Fingerprint matrix shape: {X_fps.shape}")

# ── Step 2: Variance filter ──────────────────────────────────────────────────
X_filt, keep_mask = variance_filter(X_fps)
print(f"  After variance filter   : {X_filt.shape[1]} bits retained")

# ── Step 3: PCA ─────────────────────────────────────────────────────────────
X_pca, pca = pca_reduce(X_filt, n_components=PCA_N_COMPONENTS)
cumvar = pca.explained_variance_ratio_.cumsum()[-1] * 100
print(f"  PCA {PCA_N_COMPONENTS}D cumulative variance: {cumvar:.1f}%")

In [ ]:
# ── Step 4: Train GTM (or load pre-trained) ─────────────────────────────────

if PRETRAINED_MODEL_PATH and os.path.exists(PRETRAINED_MODEL_PATH):
    print(f"Loading pre-trained GTM from: {PRETRAINED_MODEL_PATH}")
    model = GTM.load(PRETRAINED_MODEL_PATH)
    coords_ref, _ = model.transform(X_pca)   # project reference data
    print("Model loaded. Projecting ChEMBL data …")
else:
    print(f"Training GTM (grid={GTM_GRID_SIZE}×{GTM_GRID_SIZE}, rbf={GTM_RBF_SIZE}, iter={GTM_N_ITER}) …")
    print("This may take a few minutes for large datasets (>10k molecules).")
    model = GTM(grid_size=GTM_GRID_SIZE, rbf_size=GTM_RBF_SIZE, n_iter=GTM_N_ITER)
    coords_ref, _ = model.fit_transform(X_pca)
    # Save for future use
    save_path = os.path.join(OUTPUT_DIR, "gtm_reference_model.pkl")
    model.save(save_path)
    print(f"\nModel saved → {save_path}")

print(f"β (precision): {model.beta_:.4f}   σ = {1/np.sqrt(model.beta_):.4f}")
print(f"Molecules projected on {model.grid_size}×{model.grid_size} grid")

## Section 6 — Project Your Compounds

We apply **the same PCA transform** learned on ChEMBL to your compound fingerprints, then call
`model.transform()`. This is the key step that avoids training bias: the coordinate system is
defined entirely by the ChEMBL reference set. Your compounds are embedded into it, not the other way around.

In [ ]:
print(f"Computing {FP_TYPE} fingerprints for {len(my_smi):,} of your molecules …")
X_my = compute_fingerprints(my_smi, fp_type=FP_TYPE)
X_my_pca = apply_pca(X_my, pca, keep_mask=keep_mask)

coords_my, R_my = model.transform(X_my_pca)

print(f"Projected {len(coords_my):,} molecules onto the GTM map.")
print(f"  coords_ref shape : {coords_ref.shape}  (ChEMBL reference)")
print(f"  coords_my  shape : {coords_my.shape}   (your data)")

## Section 7 — Visualisations

### 7a — Activity Landscape (ChEMBL reference)

The grey background is the **density landscape** of the training set: darker regions have more
ChEMBL compounds. Coloured dots show pIC50 values (green = high potency, red = low potency).
Grey dots indicate compounds without a pIC50 value (e.g. inactive-only annotations).

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

land = model.landscape(X_pca)   # 2-D density grid

# Shared colour scale across both panels
all_acts = np.concatenate([chembl_activity, my_activity])
vmin = np.nanpercentile(all_acts, 5)
vmax = np.nanpercentile(all_acts, 95)
cmap = "RdYlGn"

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

panels = [
    (axes[0], coords_ref, chembl_activity,  "ChEMBL — pIC50 landscape",          "pIC50"),
    (axes[1], coords_my,  my_activity,       f"{MY_DATASET_LABEL} — activity map", MY_ACTIVITY_COL),
]

for ax, coords, acts, title, cb_label in panels:
    # Density landscape as background
    ax.imshow(land, origin="lower", extent=[-1, 1, -1, 1],
              cmap="Greys", aspect="auto", interpolation="bilinear",
              alpha=0.4, zorder=1)

    nan_mask = np.isnan(acts)
    if nan_mask.any():
        ax.scatter(coords[nan_mask, 0], coords[nan_mask, 1],
                   c="grey", s=18, alpha=0.4, zorder=4,
                   edgecolors="none", label="No activity value")

    sc = ax.scatter(coords[~nan_mask, 0], coords[~nan_mask, 1],
                    c=acts[~nan_mask], cmap=cmap, vmin=vmin, vmax=vmax,
                    s=20, alpha=0.85, zorder=5, edgecolors="none")

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.08)
    cb = plt.colorbar(sc, cax=cax)
    cb.set_label(cb_label, fontsize=9)

    if nan_mask.any():
        ax.legend(fontsize=8, loc="upper left", framealpha=0.7)

    ax.set_xlabel("GTM dimension 1", fontsize=11)
    ax.set_ylabel("GTM dimension 2", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold")

fig.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "gtm_activity_landscape.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Saved → {fig_path}")
plt.show()

### 7b — Overlay: Where do your compounds sit in ChEMBL chemical space?

This plot superimposes your compounds (coloured markers) on the ChEMBL density map, making it
immediately clear whether your hits fall in well-explored regions or in novel areas.

In [ ]:
fig_ov, ax_ov = plot_multi_class(
    model, X_pca,
    [coords_ref, coords_my],
    ["ChEMBL reference", MY_DATASET_LABEL],
)
fig_ov.tight_layout()
fig_path2 = os.path.join(OUTPUT_DIR, "gtm_overlay.png")
fig_ov.savefig(fig_path2, dpi=150, bbox_inches="tight")
print(f"Saved → {fig_path2}")
plt.show()

## Section 8 — Diagnostics

### What are we checking?

| Diagnostic | Meaning | What to worry about |
|-----------|---------|-------------------|
| β (precision) | Inverse noise variance of the GTM. Higher β = tighter, more confident map. | Very low β (< 1) can indicate poor training or too few PCA components. |
| Responsibility entropy | Shannon entropy of the K-dimensional responsibility vector. Low entropy = molecule maps to one specific node (good). High entropy = molecule is spread across many nodes (poor projection). | > 50% of molecules at > 80% entropy fraction suggests the map is too coarse or the data is too diverse. |
| PCA scree plot | Rate at which variance drops off. | If component 50 still explains > 0.1% variance, consider increasing `PCA_N_COMPONENTS`. |

In [ ]:
# ── β and map quality ─────────────────────────────────────────────────────────
print(f"Final β (precision) : {model.beta_:.4f}")
print(f"σ (noise in PCA sp) : {1/np.sqrt(model.beta_):.4f}")
print(f"Grid nodes K        : {model.grid_size**2}")
print(f"ChEMBL molecules N  : {len(X_pca):,}")
print(f"Avg molecules/node  : {len(X_pca)/model.grid_size**2:.1f}")

# ── Responsibility entropy ────────────────────────────────────────────────────
_, R_ref = model.transform(X_pca)
H_max = np.log(model.grid_size**2)

def entropy_fraction(R):
    H = -np.sum(R * np.log(R + 1e-12), axis=1)
    return H / H_max

H_ref = entropy_fraction(R_ref)
H_my  = entropy_fraction(R_my)

print("\n── ChEMBL projection uncertainty ──")
print(f"  Mean   : {H_ref.mean():.3f}")
print(f"  Median : {np.median(H_ref):.3f}")
print(f"  >80%   : {(H_ref > 0.80).mean()*100:.1f}%  ← uncertain (poorly placed on map)")
print(f"  >90%   : {(H_ref > 0.90).mean()*100:.1f}%  ← very uncertain")

print(f"\n── {MY_DATASET_LABEL} projection uncertainty ──")
print(f"  Mean   : {H_my.mean():.3f}")
print(f"  Median : {np.median(H_my):.3f}")
print(f"  >80%   : {(H_my > 0.80).mean()*100:.1f}%")
print(f"  >90%   : {(H_my > 0.90).mean()*100:.1f}%")

In [ ]:
# ── Diagnostic figure: scree + uncertainty maps ───────────────────────────────
var    = pca.explained_variance_ratio_
cumvar = np.cumsum(var)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Scree plot
ax = axes[0]
ax.plot(np.arange(1, len(var)+1), var * 100, lw=1.5, color="steelblue")
ax.axvline(PCA_N_COMPONENTS, color='red', ls='--', label=f'Current {PCA_N_COMPONENTS}D')
ax.set_xlabel('PCA component')
ax.set_ylabel('% variance explained')
ax.set_title('Scree plot', fontweight='bold')
ax.set_xlim(0, min(len(var), PCA_N_COMPONENTS + 30))
ax.legend(); ax.grid(True, alpha=0.3)

# Uncertainty — ChEMBL
ax = axes[1]
sc1 = ax.scatter(coords_ref[:, 0], coords_ref[:, 1],
                 c=H_ref, cmap='RdYlGn_r', vmin=0.5, vmax=1.0,
                 s=8, alpha=0.6)
plt.colorbar(sc1, ax=ax, label='Entropy fraction (1=max uncertainty)')
ax.set_title('ChEMBL — Projection uncertainty', fontweight='bold')
ax.set_xlabel('GTM dim 1'); ax.set_ylabel('GTM dim 2')

# Uncertainty — your data
ax = axes[2]
sc2 = ax.scatter(coords_my[:, 0], coords_my[:, 1],
                 c=H_my, cmap='RdYlGn_r', vmin=0.5, vmax=1.0,
                 s=20, alpha=0.8)
plt.colorbar(sc2, ax=ax, label='Entropy fraction')
ax.set_title(f'{MY_DATASET_LABEL} — Projection uncertainty', fontweight='bold')
ax.set_xlabel('GTM dim 1'); ax.set_ylabel('GTM dim 2')

fig.tight_layout()
fig_path3 = os.path.join(OUTPUT_DIR, "gtm_diagnostics.png")
fig.savefig(fig_path3, dpi=150, bbox_inches="tight")
print(f"Saved → {fig_path3}")
plt.show()

## Section 9 — Summary & Next Steps

The GTM analysis is complete. Your output files are:

| File | Description |
|------|-------------|
| `gtm_reference_model.pkl` | Trained GTM model — reuse to project new datasets without retraining |
| `gtm_activity_landscape.png` | Side-by-side pIC50 maps for ChEMBL and your data |
| `gtm_overlay.png` | Your compounds overlaid on the ChEMBL density map |
| `gtm_diagnostics.png` | PCA scree, entropy uncertainty maps |

### Interpreting the results

- **Your compounds cluster in a dense ChEMBL region** → your hits are structurally similar to
  known actives. Likelihood of true activity is high, but chemical novelty may be limited.
- **Your compounds fall in a sparse region** → novel scaffold territory. Worth pursuing if
  activity data is reliable; uncertainty maps should show *low* entropy (good projection).
- **High entropy fraction for your compounds** → the GTM map cannot confidently place them.
  This usually means the scaffold is absent from the training set. Consider retraining with a
  broader/mixed library, or report these as exploratory hits requiring experimental follow-up.

### Possible extensions

- **Multi-target comparison** — project ChEMBL actives for a second target (e.g. hERG) onto the same
  map to visualise selectivity landscape.
- **Mixed training** — if your dataset is large (> 5k compounds), retrain GTM on a 1:1 mixture of
  ChEMBL and your data to get an unbiased co-embedding.
- **Cluster analysis** — apply HDBSCAN to `coords_my` / `coords_ref` to identify chemotype clusters
  on the map.
- **Model reuse** — load `gtm_reference_model.pkl` and call `apply_pca` + `model.transform` to
  project new batches of virtual screening hits in seconds.